<div style="background:linear-gradient(90deg,#dc3545,#fd7e14,#28a745); color:white; padding:24px 32px; border-radius:8px; width:97%;">

# 🎯 Lesson 4 · 本节精华(全课收官)

</div>

<small>对应主课:[4_c_🔥_full_agent.ipynb](./4_c_🔥_full_agent.ipynb)</small>

## 🎯 一句话定位

> 把 **TODO + Files + Subagents** **合体**成一个生产级 deep agent,并接入真实搜索(Tavily)+ 摘要(GPT-4o-mini)实现 **"工具大输出 → 落盘 + 返回摘要"** 的经典模式。
>
> 🧠 **本质 = 用前 3 课的所有零件,搭一台真能干活的研究 agent**。

## 📚 你应该学到的 7 件事

| # | 概念 | 你应该记住的一句话 |
|---|---|---|
| 1 | 🧱 **三大模块组合** | **TODOs**(规划)+ **Files**(外脑)+ **Subagents**(隔离),三者咬合 |
| 2 | 🔍 **`tavily_search`** | 真实搜索 + **网页原文落盘** + **只返回摘要** —— Manus 同款模式 |
| 3 | 📝 **`summarize_webpage_content`** | 用 **轻量模型**(GPT-4o-mini)做摘要,**省钱省 token** |
| 4 | 🧘 **`think_tool`** | 战略反思工具 —— LLM 主动停下来分析、规划下一步 |
| 5 | 👨‍✈️ **Supervisor-Worker** | 父 agent 派单,子 agent 干活,完美的角色分工 |
| 6 | 🪡 **prompt 拼装** | TODO + FILE + SUBAGENT instructions **按段连接**,清晰分章 |
| 7 | 📦 **`deepagents` 包** | 把所有零件封装成 `create_deep_agent(...)`,**生产级开箱即用** |

## 🏗️ 三大模块如何**咬合**

```
                           ┌─────────────────────────────────────┐
                           │   👨‍✈️ Supervisor (主 agent)             │
                           │   ──────────────────────────────────  │
                           │   工具:                              │
                           │   • write_todos / read_todos  📋     │  ← TODO 模块
                           │   • ls / read_file / write_file 📁  │  ← Files 模块
                           │   • task                        🚀  │  ← Subagent 模块
                           │   • think_tool                  🧘  │
                           └────────────────┬────────────────────┘
                                            │ task(...)
                                            ▼
                           ┌─────────────────────────────────────┐
                           │   🧑‍🔬 Research Sub-agent              │
                           │   ──────────────────────────────────  │
                           │   工具:                              │
                           │   • tavily_search    🔍 (落盘+摘要)  │
                           │   • think_tool       🧘              │
                           └─────────────────────────────────────┘
```

**三模块对应解决三类问题**:

| 模块 | 解决的 context 问题 |
|---|---|
| 📋 **TODOs** | Context **rot**(注意力衰减)→ recitation 唤醒目标 |
| 📁 **Files** | Context **bloat**(膨胀)→ 大输出落盘,messages 只留摘要 |
| 🚀 **Subagents** | Context **clash / confusion**(冲突 / 混乱)→ 开新 context 隔离 |

> 🔑 **每个模块**精准对应**一类病**,合起来就是 deep agent 的"三件套外科手术"。

## 💎 `tavily_search` 的**生产级设计精髓**

这是本节最值得抄回家的工具设计模板:

```python
@tool
def tavily_search(query, state, tool_call_id, max_results=1, topic="general"):
    # 1. 真实搜索
    search_results = run_tavily_search(query, ...)

    # 2. 拉网页全文 → HTML → markdown → 用轻量模型摘要
    processed = process_search_results(search_results)

    # 3. 落盘 + 给主 agent 只看摘要
    files = state.get("files", {})
    for r in processed:
        files[r['filename']] = f"# {r['title']}\n## Summary\n{r['summary']}\n## Raw Content\n{r['raw_content']}"
    
    summary_text = f"🔍 Found {N} result(s):\n{每个文件的一句话摘要}\nFiles: {filenames}\n💡 Use read_file() for full details."

    return Command(update={
        "files":    files,                                  # 全文落盘
        "messages": [ToolMessage(summary_text, ...)],       # 主对话只看摘要
    })
```

**设计的 3 个高级动作**:

| 动作 | 目的 |
|---|---|
| 1️⃣ **轻量模型做摘要**(GPT-4o-mini) | 摘要质量不需要顶级模型 → **成本降一个数量级** |
| 2️⃣ **filename 由摘要模型生成** | 让文件名**语义化、可索引**(不是 `result_1.md`) |
| 3️⃣ **`ToolMessage` 内提示 `read_file()`** | **主动引导主 agent**:"想看全文?用 read_file" |

## 🧘 `think_tool`:一个**反直觉的工具**

```python
@tool
def think_tool(reflection: str) -> str:
    return f"Reflection recorded: {reflection}"
```

**它什么都不做** —— 只把 LLM 给的 reflection 字符串原样返回。

**那它存在的意义是什么?**

- 🚦 **强制 LLM 停下来思考**:不调它就一直冲下一个 search
- 📝 **把思考过程显式写进 messages**:后续 LLM 能"看到"自己刚才想的东西
- 🪞 **类似 `read_todos`** —— 是"自我提示工具",不获取新信息

回到 `RESEARCHER_INSTRUCTIONS`:

> *"**CRITICAL: Use think_tool after each search to reflect on results and plan next steps**"*

**System prompt 强制 LLM 每次搜索后调 `think_tool`** —— 这就是让 "思考" 变成可观察的 trace 步骤,而不是埋在 LLM 内部黑箱里。

> 💡 **设计哲学**:**让 LLM 的思考过程留痕** —— 既方便调试,也强制 LLM 慢下来。

## 📦 `deepagents` 包 —— 从此一行搞定

学完本课,你完全理解了底层。**实战可以直接用** [`deepagents`](https://github.com/hwchase17/deepagents):

```python
from deepagents import create_deep_agent

agent = create_deep_agent(
    tools=[tavily_search, think_tool],       # 你自己的工具
    system_prompt=INSTRUCTIONS,
    subagents=[research_sub_agent],          # 你自己定的子 agent
    model=model,
)
```

**它已自动内置**:文件系统工具、todo 工具、task 派发工具。你**只需要提供你的领域工具 + 子 agent 配置**。

<div style="background:#fff8e1; padding:10px 24px; border-left:4px solid #ff9800; border-radius:4px; width:97%;">

> 💡 **学完底层 ≠ 不用包**。学完底层 = **你能用包,但又不被包黑盒坑** —— 出问题能下沉到具体机制排查,而不是"试试重启大法"。

</div>

## 🌍 真实产品对应 + 一句话

| 你学的 | 现实里的实现 |
|---|---|
| 🎯 全栈 deep agent | **Claude Code / Devin / Cursor Composer** 的内部架构 |
| 🔍 大输出 → 落盘 + 摘要 | **Perplexity Pro Search / Claude Search**(读网页 → 总结) |
| 🧘 think_tool | Anthropic 公开的 [extended thinking](https://docs.anthropic.com/en/docs/build-with-claude/extended-thinking) 思路的轻量版 |
| 📦 deepagents 包 | LangChain 官方推荐的 "deep agent" 起点 |

## ✨ 一句话带走 —— 全课总结

<div style="background:#e7f7e9; padding:10px 24px; border-left:4px solid #28a745; border-radius:4px; width:97%;">

> 🔑 **Deep agent = ReAct 循环 + 3 件武器**:
>
> - 📋 **TODOs** 治 "忘"
> - 📁 **Files**  治 "挤"
> - 🚀 **Subagents** 治 "乱"
>
> 每件武器精准对一类 context 病。**合起来 = 一台能跑 50+ 工具调用还不崩溃的研究 agent**。
>
> 学完这一节,你具备了**自己造 / 看穿任何 deep agent 框架**的能力。

</div>